In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

In [2]:
wilayah_miskin = pd.read_excel("Dataset_Final.xlsx", sheet_name="Dataset_114")
var = ["PPM", "TPAK", "TPT", "PDRB", "KP"]
X_prediktor = wilayah_miskin[var]
Y_target = wilayah_miskin["IPM"]
grup = wilayah_miskin["Kabupaten/Kota"]

def hitung_metrik(y_asli, y_pred):
    return {"MAE": mean_absolute_error(y_asli, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_asli, y_pred)),
            "R2": r2_score(y_asli, y_pred)}

In [3]:
def buat_model(param, y_latih):
    dasar = dict(objective="reg:squarederror", tree_method="exact",
                 reg_lambda=1, reg_alpha=0, gamma=0, random_state=0)
    dasar.update(param)
    return XGBRegressor(base_score=float(y_latih.mean()), **dasar)

PARAM_UTAMA = {"n_estimators": 100, "max_depth": 2, "learning_rate": 0.04, "min_child_weight": 5}

In [4]:
pembagi = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
idx_latih, idx_uji = next(pembagi.split(X_prediktor, Y_target, groups=grup))
data_latih = wilayah_miskin.iloc[idx_latih]
X_latih, y_latih = X_prediktor.iloc[idx_latih], Y_target.iloc[idx_latih]
X_uji, y_uji = X_prediktor.iloc[idx_uji], Y_target.iloc[idx_uji]
print(f"Latih: {grup.iloc[idx_latih].nunique()} daerah, {len(idx_latih)} baris | "
      f"Uji: {grup.iloc[idx_uji].nunique()} daerah, {len(idx_uji)} baris")

model_utama = buat_model(PARAM_UTAMA, y_latih).fit(X_latih, y_latih)
metrik_utama = hitung_metrik(y_uji, model_utama.predict(X_uji))
print({k: round(v, 3) for k, v in metrik_utama.items()})

Latih: 91 daerah, 819 baris | Uji: 23 daerah, 207 baris
{'MAE': 2.611, 'RMSE': 3.388, 'R2': 0.834}


In [5]:
hasil_rasio = []
for porsi_uji in [0.15, 0.20, 0.25, 0.30]:
    for seed in range(10):
        pembagi_r = GroupShuffleSplit(n_splits=1, test_size=porsi_uji, random_state=seed)
        idx_l, idx_u = next(pembagi_r.split(X_prediktor, Y_target, groups=grup))
        y_l, y_u = Y_target.iloc[idx_l], Y_target.iloc[idx_u]

        model_r = buat_model(PARAM_UTAMA, y_l).fit(X_prediktor.iloc[idx_l], y_l)
        pred = model_r.predict(X_prediktor.iloc[idx_u])
        mae_baseline = mean_absolute_error(y_u, np.full(len(y_u), y_l.mean()))

        hasil_rasio.append({"Rasio": f"{round((1 - porsi_uji) * 100)}/{round(porsi_uji * 100)}",
                            "Seed": seed,
                            "Daerah uji": grup.iloc[idx_u].nunique(),
                            **hitung_metrik(y_u, pred),
                            "MAE baseline": mae_baseline})

tabel_rasio = pd.DataFrame(hasil_rasio)
ringkasan_rasio = tabel_rasio.groupby("Rasio").agg(
    Daerah_uji=("Daerah uji", "first"),
    MAE_rata=("MAE", "mean"), MAE_sd=("MAE", "std"),
    RMSE_rata=("RMSE", "mean"),
    R2_rata=("R2", "mean"), R2_sd=("R2", "std"),
    MAE_baseline=("MAE baseline", "mean"),
).round(3)
ringkasan_rasio

,Daerah_uji,MAE_rata,MAE_sd,RMSE_rata,R2_rata,R2_sd,MAE_baseline
Rasio,,,,,,,
70/30,35,3.224,0.313,4.366,0.622,0.077,5.207
75/25,29,3.119,0.528,4.141,0.642,0.049,5.116
80/20,23,2.846,0.484,3.679,0.684,0.062,5.004
85/15,18,2.816,0.718,3.547,0.690,0.114,4.913


In [6]:
grid_gabungan = {
    "n_estimators": [100, 200, 300, 500, 1000],
    "max_depth": [2, 3, 4, 6, 8, 10],
    "learning_rate": [0.01, 0.04, 0.1],
    "min_child_weight": [1, 3, 5],
    "reg_lambda": [0, 1, 5, 10],
    "reg_alpha": [0, 0.1, 1],
}
jumlah = int(np.prod([len(v) for v in grid_gabungan.values()]))
print(f"{jumlah} kombinasi x 5 fold = {jumlah * 5} model")

pencarian_gab = GridSearchCV(buat_model({"n_jobs": 1}, y_latih), grid_gabungan,
                             cv=GroupKFold(n_splits=5), scoring="neg_mean_absolute_error",
                             n_jobs=-1, verbose=1)
pencarian_gab.fit(X_latih, y_latih, groups=data_latih["Kabupaten/Kota"])

3240 kombinasi x 5 fold = 16200 model
Fitting 5 folds for each of 3240 candidates, totalling 16200 fits


GridSearchCV(cv=GroupKFold(n_splits=5),
             estimator=XGBRegressor(base_score=64.23161172161173, booster=None,
                                    callbacks=None, colsample_bylevel=None,
                                    colsample_bynode=None,
                                    colsample_bytree=None, device=None,
                                    early_stopping_rounds=None,
                                    enable_categorical=True, eval_metric=None,
                                    feature_types=None, feature_weights=None,
                                    gamma=0, grow_policy=None,
                                    importance_type=Non...
                                    min_child_weight=None, missing=nan,
                                    monotone_constraints=None,
                                    multi_strategy=None, n_estimators=None,
                                    n_jobs=1, num_parallel_tree=None, ...),
             n_jobs=-1,
             param_grid={'learning_rate': [0.01, 0.04, 0.1],
                         'max_depth': [2, 3, 4, 6, 8, 10],
                         'min_child_weight': [1, 3, 5],
                         'n_estimators': [100, 200, 300, 500, 1000],
                         'reg_alpha': [0, 0.1, 1],
                         'reg_lambda': [0, 1, 5, 10]},
             scoring='neg_mean_absolute_error', verbose=1)

In [7]:
hasil_gab = pd.DataFrame(pencarian_gab.cv_results_)
nama_param = list(grid_gabungan)
hasil_gab = hasil_gab[[f"param_{p}" for p in nama_param] + ["mean_test_score", "std_test_score"]]
hasil_gab.columns = nama_param + ["MAE validasi", "SD antar fold"]
hasil_gab["MAE validasi"] = -hasil_gab["MAE validasi"]
hasil_gab = hasil_gab.sort_values("MAE validasi").reset_index(drop=True)
hasil_gab.index = hasil_gab.index + 1
hasil_gab.index.name = "Peringkat"

param_utama_lengkap = {**PARAM_UTAMA, "reg_lambda": 1, "reg_alpha": 0}
cocok = np.ones(len(hasil_gab), dtype=bool)
for p, nilai in param_utama_lengkap.items():
    cocok &= hasil_gab[p].astype(float) == float(nilai)
peringkat_utama = hasil_gab.index[cocok][0]

print("Terbaik:", pencarian_gab.best_params_)
print(f"Peringkat model utama: {peringkat_utama} dari {len(hasil_gab)}")
hasil_gab.head(10).round(3)

Terbaik: {'learning_rate': 0.04, 'max_depth': 2, 'min_child_weight': 5, 'n_estimators': 100, 'reg_alpha': 0.1, 'reg_lambda': 1}
Peringkat model utama: 4 dari 3240


,n_estimators,max_depth,learning_rate,min_child_weight,reg_lambda,reg_alpha,MAE validasi,SD antar fold
Peringkat,,,,,,,,
1,100,2,0.04,5,1,0.1,3.412,0.539
2,100,2,0.04,1,1,0.1,3.413,0.539
3,100,2,0.04,3,1,0.1,3.413,0.539
4,100,2,0.04,5,1,0,3.414,0.543
5,100,2,0.04,3,1,0,3.414,0.544
6,100,2,0.04,1,1,0,3.414,0.544
7,100,2,0.04,3,1,1,3.419,0.541
8,100,2,0.04,1,1,1,3.419,0.541
9,100,2,0.04,5,1,1,3.422,0.544


In [8]:
param_gab = pencarian_gab.best_params_
model_gab = buat_model(param_gab, y_latih).fit(X_latih, y_latih)

mae_terbaik = hasil_gab.loc[1, "MAE validasi"]
sd_terbaik = hasil_gab.loc[1, "SD antar fold"]
mae_utama_val = hasil_gab.loc[peringkat_utama, "MAE validasi"]

perbandingan_gab = pd.DataFrame({
    "Model utama": {"MAE validasi": mae_utama_val, **metrik_utama},
    "Terbaik pencarian gabungan": {"MAE validasi": mae_terbaik,
                                   **hitung_metrik(y_uji, model_gab.predict(X_uji))},
}).round(3)
print(f"Selisih MAE validasi: {mae_utama_val - mae_terbaik:.3f} | SD antar fold: {sd_terbaik:.3f}")
print("Model final perlu diganti?", (mae_utama_val - mae_terbaik) > sd_terbaik)
perbandingan_gab

Selisih MAE validasi: 0.002 | SD antar fold: 0.539
Model final perlu diganti? False


,Model utama,Terbaik pencarian gabungan
MAE validasi,3.414,3.412
MAE,2.611,2.604
RMSE,3.388,3.383
R2,0.834,0.834


In [9]:
ringkasan_rasio.to_csv("hasil_E1_rasio_split.csv")
tabel_rasio.to_csv("hasil_E1_rasio_split_detail.csv", index=False)
hasil_gab.to_csv("hasil_E2_tuning_gabungan.csv")
perbandingan_gab.to_csv("hasil_E2_perbandingan.csv")

In [ ]:
from sklearn.linear_model import LinearRegression

kfold = GroupKFold(n_splits=5)
MODEL = ["Baseline", "Regresi linear", "XGBoost (model utama)"]
pred_silang = {nama: np.zeros(len(Y_target)) for nama in MODEL}
hasil_fold_banding = []

for i, (idx_l, idx_v) in enumerate(kfold.split(X_prediktor, Y_target, groups=grup), start=1):
    X_l, y_l = X_prediktor.iloc[idx_l], Y_target.iloc[idx_l]
    X_v, y_v = X_prediktor.iloc[idx_v], Y_target.iloc[idx_v]
    prediksi = {
        "Baseline": np.full(len(idx_v), y_l.mean()),
        "Regresi linear": LinearRegression().fit(X_l, y_l).predict(X_v),
        "XGBoost (model utama)": buat_model(PARAM_UTAMA, y_l).fit(X_l, y_l).predict(X_v),
    }
    for nama, p in prediksi.items():
        pred_silang[nama][idx_v] = p
        hasil_fold_banding.append({"Fold": i, "Model": nama, **hitung_metrik(y_v, p)})

tabel_fold_banding = pd.DataFrame(hasil_fold_banding)
model_linear = LinearRegression().fit(X_latih, y_latih)
pred_uji = {"Baseline": np.full(len(y_uji), y_latih.mean()),
            "Regresi linear": model_linear.predict(X_uji),
            "XGBoost (model utama)": model_utama.predict(X_uji)}

baris = []
for nama in MODEL:
    uji = hitung_metrik(y_uji, pred_uji[nama])
    silang = hitung_metrik(Y_target, pred_silang[nama])
    fold = tabel_fold_banding[tabel_fold_banding["Model"] == nama]
    baris.append({"Model": nama,
                  "Uji MAE": uji["MAE"], "Uji RMSE": uji["RMSE"], "Uji R2": uji["R2"],
                  "Silang MAE": silang["MAE"], "Silang RMSE": silang["RMSE"], "Silang R2": silang["R2"],
                  "MAE fold rata-rata": fold["MAE"].mean(), "MAE fold SD": fold["MAE"].std()})
perbandingan_model = pd.DataFrame(baris).set_index("Model").round(3)

mae_fold = tabel_fold_banding.pivot(index="Fold", columns="Model", values="MAE")
terbaik = int((mae_fold["XGBoost (model utama)"] < mae_fold["Regresi linear"]).sum())
print(f"XGBoost lebih baik dari regresi linear di {terbaik} dari 5 fold")
display(mae_fold.round(3))
perbandingan_model

XGBoost lebih baik dari regresi linear di 4 dari 5 fold


Model,Baseline,Regresi linear,XGBoost (model utama)
Fold,,,
1,4.247,3.450,2.541
2,6.053,4.012,3.239
3,6.081,4.069,3.392
4,5.423,3.313,3.387
5,4.963,3.129,2.981


,Uji MAE,Uji RMSE,Uji R2,Silang MAE,Silang RMSE,Silang R2,MAE fold rata-rata,MAE fold SD
Model,,,,,,,,
Baseline,6.500,8.390,-0.019,5.357,7.490,-0.026,5.353,0.775
Regresi linear,2.901,3.709,0.801,3.599,4.868,0.567,3.594,0.423
XGBoost (model utama),2.611,3.388,0.834,3.109,4.155,0.684,3.108,0.358


In [11]:
koefisien = pd.DataFrame({"Indikator": var, "Koefisien": model_linear.coef_})
koefisien["Arah"] = np.where(koefisien["Koefisien"] > 0, "Positif", "Negatif")
print("Intersep:", round(model_linear.intercept_, 3))
koefisien

Intersep: 92.867


,Indikator,Koefisien,Arah
0,PPM,-0.588148,Negatif
1,TPAK,-0.233152,Negatif
2,TPT,0.171286,Positif
3,PDRB,0.000018,Positif
4,KP,0.002561,Positif


In [12]:
perbandingan_model.to_csv("hasil_E3_perbandingan_model.csv")
tabel_fold_banding.to_csv("hasil_E3_per_fold.csv", index=False)
koefisien.to_csv("hasil_E3_koefisien_linear.csv", index=False)

In [13]:
skenario_var = {"Semua variabel (model utama)": var}
for v in var:
    skenario_var[f"Tanpa {v}"] = [x for x in var if x != v]
skenario_var["Hanya PPM"] = ["PPM"]

hasil_var = []
for nama, kolom in skenario_var.items():
    pred = np.zeros(len(Y_target))
    for i, (idx_l, idx_v) in enumerate(kfold.split(X_prediktor, Y_target, groups=grup), start=1):
        X_l = wilayah_miskin[kolom].iloc[idx_l]
        y_l = Y_target.iloc[idx_l]
        model_v = buat_model(PARAM_UTAMA, y_l).fit(X_l, y_l)
        p = model_v.predict(wilayah_miskin[kolom].iloc[idx_v])
        pred[idx_v] = p
        hasil_var.append({"Skenario": nama, "Fold": i, **hitung_metrik(Y_target.iloc[idx_v], p)})
    hasil_var.append({"Skenario": nama, "Fold": "Gabungan", **hitung_metrik(Y_target, pred)})

tabel_var = pd.DataFrame(hasil_var)
per_fold = tabel_var[tabel_var["Fold"] != "Gabungan"].pivot(index="Fold", columns="Skenario", values="MAE")
gabungan = tabel_var[tabel_var["Fold"] == "Gabungan"].set_index("Skenario")[["MAE", "RMSE", "R2"]]

acuan = per_fold["Semua variabel (model utama)"]
ringkasan_var = gabungan.copy()
ringkasan_var["Selisih MAE vs model utama"] = gabungan["MAE"] - gabungan.loc["Semua variabel (model utama)", "MAE"]
ringkasan_var["Fold lebih buruk"] = [int((per_fold[s] > acuan).sum()) for s in ringkasan_var.index]
ringkasan_var = ringkasan_var.sort_values("MAE").round(3)
ringkasan_var

,MAE,RMSE,R2,Selisih MAE vs model utama,Fold lebih buruk
Skenario,,,,,
Tanpa TPAK,3.050,4.107,0.692,-0.059,2
Semua variabel (model utama),3.109,4.155,0.684,0.000,0
Tanpa TPT,3.210,4.312,0.660,0.101,5
Tanpa KP,3.265,4.409,0.645,0.156,4
Tanpa PDRB,3.329,4.450,0.638,0.220,4
Tanpa PPM,3.470,4.750,0.588,0.361,5
Hanya PPM,3.615,4.970,0.548,0.506,4


In [ ]:
latih_t = wilayah_miskin[wilayah_miskin["Tahun"] <= 2024]
uji_t = wilayah_miskin[wilayah_miskin["Tahun"] == 2025]
X_lt, y_lt = latih_t[var], latih_t["IPM"]
X_ut, y_ut = uji_t[var], uji_t["IPM"]

ipm_2024 = (wilayah_miskin[wilayah_miskin["Tahun"] == 2024]
            .set_index("Kabupaten/Kota")["IPM"])
pred_waktu = {
    "Baseline (rata-rata 2017-2024)": np.full(len(y_ut), y_lt.mean()),
    "IPM 2025 = IPM 2024": uji_t["Kabupaten/Kota"].map(ipm_2024).values,
    "Regresi linear": LinearRegression().fit(X_lt, y_lt).predict(X_ut),
    "XGBoost (parameter utama)": buat_model(PARAM_UTAMA, y_lt).fit(X_lt, y_lt).predict(X_ut),
}
baris_waktu = []
for nama, p in pred_waktu.items():
    baris_waktu.append({"Model": nama, **hitung_metrik(y_ut, p),
                        "Rata-rata selisih (aktual - prediksi)": float(np.mean(y_ut.values - p))})
hasil_waktu = pd.DataFrame(baris_waktu).set_index("Model").round(3)
print(f"Latih: {len(latih_t)} baris (2017-2024) | Uji: {len(uji_t)} baris (2025)")
hasil_waktu

Latih: 912 baris (2017–2024) | Uji: 114 baris (2025)


,MAE,RMSE,R2,Rata-rata selisih (aktual − prediksi)
Model,,,,
Baseline (rata-rata 2017–2024),6.278,7.551,-0.166,2.852
IPM 2025 = IPM 2024,0.744,0.768,0.988,0.744
Regresi linear,3.172,4.140,0.649,1.528
XGBoost (parameter utama),2.747,3.515,0.747,1.771


In [ ]:
daerah_latih = set(data_latih["Kabupaten/Kota"])
latih_ketat = latih_t[latih_t["Kabupaten/Kota"].isin(daerah_latih)]
uji_ketat = uji_t[~uji_t["Kabupaten/Kota"].isin(daerah_latih)]
model_ketat = buat_model(PARAM_UTAMA, latih_ketat["IPM"]).fit(latih_ketat[var], latih_ketat["IPM"])
p_ketat = model_ketat.predict(uji_ketat[var])
hasil_ketat = pd.DataFrame([
    {"Skenario": "Tahun baru, daerah sudah dikenal", **hitung_metrik(y_ut, pred_waktu["XGBoost (parameter utama)"])},
    {"Skenario": "Tahun baru dan daerah baru", **hitung_metrik(uji_ketat["IPM"], p_ketat),
     "Daerah": uji_ketat["Kabupaten/Kota"].nunique()},
]).set_index("Skenario").round(3)
print(f"Latih: {latih_ketat['Kabupaten/Kota'].nunique()} daerah x 2017-2024 | Uji: {uji_ketat['Kabupaten/Kota'].nunique()} daerah lain, 2025")
hasil_ketat

Latih: 91 daerah × 2017–2024 | Uji: 23 daerah lain, 2025


,MAE,RMSE,R2,Daerah
Skenario,,,,
"Tahun baru, daerah sudah dikenal",2.747,3.515,0.747,NaN
Tahun baru dan daerah baru,3.020,3.705,0.791,23.0


In [16]:
ringkasan_var.to_csv("hasil_E4_hapus_variabel.csv")
tabel_var.to_csv("hasil_E4_per_fold.csv", index=False)
hasil_waktu.to_csv("hasil_E5_uji_2025.csv")
hasil_ketat.to_csv("hasil_E5_uji_2025_ketat.csv")